# Teacher-Supervised Formative Mathematics Assessment

**F1 and F2.1 | Reproducible Jupyter walkthrough**

This notebook uses the Python modules shipped in the repository. It demonstrates system-level functionality using invented cases and does not contact a model unless explicitly enabled. All generated feedback must be reviewed by an educator.


## 1. Locate the repository and load the modules

Launch Jupyter from the repository root or its `notebooks/` directory.


In [ ]:
from pathlib import Path
import sys
import json
import subprocess

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent) if (p / "chapter9_system").is_dir() and (p / "chapter9_f21").is_dir()), None)
assert ROOT is not None, "Open this notebook from the repository root or notebooks/ folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])


## 2. Automated regression tests

All tests operate offline. One historical-data test is expected to be skipped because restricted data is not included in the public repository.


In [ ]:
command = [sys.executable, "-m", "unittest", "discover", "-s", ".", "-t", ".", "-p", "test_*.py", "-v"]
result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Automated regression tests failed."
print("Regression tests passed.")


## 3. Synthetic demonstration

The fixture provides deterministic, prewritten diagnostic and feedback responses. This tests pipeline integration, **not** diagnostic correctness or educational effectiveness.


In [ ]:
from chapter9_system.contracts import Case
from chapter9_system.pipeline import assess as assess_f1
from chapter9_f21.pipeline import assess as assess_f21

case = Case(
    case_id="SYNTHETIC_001",
    question="Mira has 8 markers and receives 5 more. How many markers does Mira have?",
    student_solution="I worked out 8 + 5 = 12.",
)

class OfflineFixture:
    model = "synthetic-fixture"
    def __init__(self):
        self.calls = 0
    def ask(self, messages):
        self.calls += 1
        structured = "target_quote" in messages[0]["content"]
        if self.calls == 1:
            diagnosis = {
                "error_status": "possible",
                "evidence_quote": "8 + 5 = 12",
                "diagnosis": "The student may have made an addition error.",
                "strategy": "step_check",
                "arithmetic_expression": "8+5",
            }
            if structured:
                diagnosis["target_quote"] = "How many markers does Mira have?"
            return {"parsed": diagnosis, "seconds": 0.0}
        return {"parsed": {
            "teacher_draft": "The learner may benefit from checking the addition step.",
            "scaffold_question": "How could the learner verify the sum of eight and five?",
        }, "seconds": 0.0}

f1 = assess_f1(case, OfflineFixture())
f21 = assess_f21(case, OfflineFixture())
for label, output in (("F1", f1), ("F2.1", f21)):
    assert output["automated_personalised_feedback_authorised"] is False
    assert output["model_calls"] == 2
    print(label, "| route:", output["route"])
    print("Diagnosis:", (output.get("diagnosis") or {}).get("diagnosis"))
    print("Calculation:", output["calculation"])
    print("Teacher draft:", (output.get("feedback_draft") or {}).get("teacher_draft"))
    print()


## 4. Optional inference with a local model

A running Ollama service and the `qwen3:4b-instruct` model are required. This cell is disabled by default. It writes into an isolated synthetic-example result directory; it does not access any held-out evaluation data.


In [ ]:
RUN_LOCAL_EXAMPLE = False
if RUN_LOCAL_EXAMPLE:
    from chapter9_system.client import OllamaClient
    from chapter9_f21.client import StructuredOllamaClient
    from chapter9_system.experiments import run_cases
    from chapter9_f21.experiments import run_f21

    output_root = ROOT / "results" / "synthetic_local"
    summary_f1 = run_cases(
        [case], OllamaClient(model="qwen3:4b-instruct"),
        output_root / "f1_v1", max_cases=1,
    )
    summary_f21 = run_f21(
        [case], StructuredOllamaClient(model="qwen3:4b-instruct"),
        output_root / "f21_v1", max_cases=1,
    )
    print(json.dumps({"F1": summary_f1, "F2.1": summary_f21}, indent=2))
else:
    print("Optional local inference is disabled.")


## 5. Independent evaluation: operational indicators

The values below summarize the completed 76-problem paired assessment. Restricted problems, individual predictions, reference answers, and reviewer annotations are not included. **Coverage is not a measure of mathematical correctness.**


In [ ]:
from math import comb
summary = {
    "F1":   {"cases": 76, "diagnoses": 70, "drafts": 70, "calls": 145, "mean_seconds": 91.420},
    "F2.1": {"cases": 76, "diagnoses": 76, "drafts": 76, "calls": 152, "mean_seconds": 104.338},
}
for method, metrics in summary.items():
    print(f"{method}: {metrics['drafts']}/{metrics['cases']} draft coverage; "
          f"mean runtime {metrics['mean_seconds']:.3f} s/case")
# Exact two-sided McNemar test: six discordant pairs, all favoring F2.1.
n = 6
p_value = 2 * sum(comb(n, k) for k in range(1)) / 2**n
print("Exact McNemar p-value for output coverage:", round(p_value, 5))
print("Mathematical correctness was not verified by these operational indicators.")


## 6. Interpretation and limits

No system output is eligible for unattended delivery to a student. The arithmetic checks are intentionally limited, and expert review identified substantial mathematical and pedagogical errors. Consult `docs/EVALUATION.md` and `docs/REPRODUCIBILITY.md` for the evidence boundaries and the research protocol.
